# Explorar la API de Resultados Electorales

Primer caso de prueba: **2011, Buenos Aires (distrito), La Plata (sección provincial), Generales, categoría 1**.

Paso 1: traer el JSON crudo tal cual viene de la API y mirarlo. Paso 2 (más abajo): recién ahí instanciar el objeto de dominio.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import json

from electoral.client import ResultadosClient
from electoral.models import ResultadoElectoral

client = ResultadosClient(cache_dir=Path.cwd().parent / "data" / "distrito")

## 1. JSON crudo

`distritoId=2` = Buenos Aires, `seccionProvincialId=8` = Sección Capital, `seccionId=63` = La Plata, `categoriaId=1` = PRESIDENTE. Estos cuatro valores ya no son una suposición: están confirmados contra `la_plata_2011.csv` (la descarga oficial del sitio), que trae explícitamente `seccionprovincial_id=8`/`seccionprovincial_nombre=Sección Capital`, `seccion_id=63`/`seccion_nombre=La Plata` y `cargo_id=1`/`cargo_nombre=PRESIDENTE` en cada fila.

In [2]:
consulta = dict(
    anio_eleccion=2011,
    categoria_nombre="presidente/generales",  # organiza el caché en data/distrito/2011/presidente/generales/
    tipo_eleccion=2,  # Generales
    categoria_id=1,  # PRESIDENTE
    distrito_id=2,  # Buenos Aires
    seccion_provincial_id=8,  # Sección Capital
    seccion_id=63,  # La Plata
)

raw = client.get_resultados(**consulta)
print(json.dumps(raw, indent=2, ensure_ascii=False))

{
  "fechaTotalizacion": "2023-11-25T20:12:26.043Z",
  "estadoRecuento": {
    "mesasEsperadas": 0,
    "mesasTotalizadas": 1338,
    "mesasTotalizadasPorcentaje": 0,
    "cantidadElectores": 455794,
    "cantidadVotantes": 369821,
    "participacionPorcentaje": 81.14
  },
  "valoresTotalizadosPositivos": [
    {
      "idAgrupacion": "0133",
      "nombreAgrupacion": "Alianza Compromiso Federal                                                                          ",
      "votos": 30978,
      "votosPorcentaje": 8.73,
      "idAgrupacionTelegrama": "",
      "urlLogo": ""
    },
    {
      "idAgrupacion": "0135",
      "nombreAgrupacion": "Alianza Frente de Izquierda y de los Trabajadores                                                   ",
      "votos": 12654,
      "votosPorcentaje": 3.57,
      "idAgrupacionTelegrama": "",
      "urlLogo": ""
    },
    {
      "idAgrupacion": "0047",
      "nombreAgrupacion": "Coalición Cívica - Afirmación para una República Igualitaria ARI  

In [3]:
# Cosas a mirar a ojo en el JSON crudo:
# - keys de primer nivel
# - tipo real de idAgrupacion (str vs int, según el año)
# - si nombreAgrupacion viene con padding de espacios
# - si valoresTotalizadosOtros tiene más campos que votosNulos/votosEnBlanco
print("keys top:", list(raw.keys()))
print("fechaTotalizacion:", raw["fechaTotalizacion"])
print("keys valoresTotalizadosOtros:", list(raw["valoresTotalizadosOtros"].keys()))
primero = raw["valoresTotalizadosPositivos"][0]
print("primer agrupacion:", primero)
print("tipo de idAgrupacion:", type(primero["idAgrupacion"]))

keys top: ['fechaTotalizacion', 'estadoRecuento', 'valoresTotalizadosPositivos', 'valoresTotalizadosOtros']
fechaTotalizacion: 2023-11-25T20:12:26.043Z
keys valoresTotalizadosOtros: ['votosNulos', 'votosNulosPorcentaje', 'votosEnBlanco', 'votosEnBlancoPorcentaje', 'votosRecurridosComandoImpugnados', 'votosRecurridosComandoImpugnadosPorcentaje']
primer agrupacion: {'idAgrupacion': '0133', 'nombreAgrupacion': 'Alianza Compromiso Federal                                                                          ', 'votos': 30978, 'votosPorcentaje': 8.73, 'idAgrupacionTelegrama': '', 'urlLogo': ''}
tipo de idAgrupacion: <class 'str'>


## 2. Objeto de dominio

Se parsea el JSON crudo. `ResultadoElectoral.from_json` recibe también `consulta` para que el objeto sepa con qué parámetros se pidió (la API no lo indica en la respuesta).

In [4]:
resultado = ResultadoElectoral.from_json(raw, consulta=consulta)
resultado

ResultadoElectoral(fecha_totalizacion='2023-11-25T20:12:26.043Z', estado_recuento=EstadoRecuento(mesas_esperadas=0, mesas_totalizadas=1338, mesas_totalizadas_porcentaje=0, cantidad_electores=455794, cantidad_votantes=369821, participacion_porcentaje=81.14), valores_totalizados_positivos=[ValorAgrupacion(id_agrupacion='0133', nombre_agrupacion='Alianza Compromiso Federal', votos=30978, votos_porcentaje=8.73, listas=[]), ValorAgrupacion(id_agrupacion='0135', nombre_agrupacion='Alianza Frente de Izquierda y de los Trabajadores', votos=12654, votos_porcentaje=3.57, listas=[]), ValorAgrupacion(id_agrupacion='0047', nombre_agrupacion='Coalición Cívica - Afirmación para una República Igualitaria ARI', votos=9092, votos_porcentaje=2.56, listas=[]), ValorAgrupacion(id_agrupacion='0132', nombre_agrupacion='Alianza Frente Popular', votos=27275, votos_porcentaje=7.69, listas=[]), ValorAgrupacion(id_agrupacion='0137', nombre_agrupacion='Alianza Unión para el Desarrollo Social', votos=42476, votos_p

In [5]:
resultado.estado_recuento

EstadoRecuento(mesas_esperadas=0, mesas_totalizadas=1338, mesas_totalizadas_porcentaje=0, cantidad_electores=455794, cantidad_votantes=369821, participacion_porcentaje=81.14)

In [6]:
for agrupacion in sorted(
    resultado.valores_totalizados_positivos, key=lambda v: v.votos, reverse=True
):
    print(f"{agrupacion.votos_porcentaje:5.2f}%  {agrupacion.votos:>7}  {agrupacion.nombre_agrupacion}")

43.29%   153609  Alianza Frente para la Victoria
22.19%    78714  Alianza Frente Amplio Progresista
11.97%    42476  Alianza Unión para el Desarrollo Social
 8.73%    30978  Alianza Compromiso Federal
 7.69%    27275  Alianza Frente Popular
 3.57%    12654  Alianza Frente de Izquierda y de los Trabajadores
 2.56%     9092  Coalición Cívica - Afirmación para una República Igualitaria ARI


## 3. Campos no modelados

Cada nivel guarda en `.extra` lo que no está modelado explícitamente. Si algo aparece acá, es una señal de que la API tiene un campo nuevo/no documentado que valdría la pena promover a campo real del modelo.

In [7]:
print("resultado.extra:", resultado.extra)
print("estado_recuento.extra:", resultado.estado_recuento.extra)
print("valores_totalizados_otros.extra:", resultado.valores_totalizados_otros.extra)
print("agrupacion[0].extra:", resultado.valores_totalizados_positivos[0].extra)

resultado.extra: {}
estado_recuento.extra: {}
valores_totalizados_otros.extra: {}
agrupacion[0].extra: {'idAgrupacionTelegrama': '', 'urlLogo': ''}
